## Loading Camera Trap Vehicle Classifier weights into PyTorch and compiling to Torchscript

Things to note: 
- the model was trained on a GPU so we need to load weights and re-compile to CPU
- it's important to check what version of torchvision (if used here) and torch you're running in this notebook environment & be sure they match the versions pinned in the deployment container's Dockerfile
- The expected input is 488x488
- The model was trained on PyTorch Lightning, which structures checkpoints a bit differently than PyTorch, so there are few additional steps required to load the model properly.

In [1]:
import torch
import timm

/opt/homebrew/Caskroom/miniforge/base/envs/camera-trap-vehicle-classifier/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# load model and weights

checkpoint_path = "./original-model/camera-trap-vehicle-classifier.2025.07.09.ckpt"
model_name = "timm/eva02_large_patch14_448.mim_m38m_ft_in22k_in1k"
num_classes = 4
device = torch.device('cpu')

timm_model_name = model_name[5:]  # Remove 'timm/' prefix
model = timm.create_model(timm_model_name, pretrained=True, num_classes=num_classes)

checkpoint = torch.load(checkpoint_path, map_location=device)
state_dict = checkpoint['state_dict']

# Remove 'model.' prefix if present (NOTE: this is necessary for loading PyTorch Lightning checkpoints)
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith('model.'):
        new_state_dict[k[6:]] = v  # strip 'model.' (6 chars)
    else:
        new_state_dict[k] = v

model.load_state_dict(new_state_dict, strict=True)
model.to(device)
model.eval() # set model to evaluation mode (disables dropout, batchnorm, etc.)

Eva(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (rope): RotaryEmbeddingCat()
  (norm_pre): Identity()
  (blocks): ModuleList(
    (0-23): 24 x EvaBlock(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): EvaAttention(
        (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path1): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): SwiGLU(
        (fc1_g)

In [ ]:
# Test the model with a dummy input
dummy = torch.randn(1, 3, 448, 448)
out = model(dummy)
print(out.shape)

torch.Size([1, 4])


In [4]:
# Save out the whole model for future inference deployment
# https://pytorch.org/tutorials/beginner/saving_loading_models.html

compiled_path = './exported-model/camera-trap-vehicle-classifier_compiled_cpu.pt'

model_scripted = torch.jit.script(model) # Export to TorchScript
model_scripted.save(compiled_path) # Save